# 🎙️ Clonador de Voz con IA (Qwen3-TTS) — script1clic.com

**Funciona con GPU o CPU.** Con GPU (`Entorno de ejecución` → `Cambiar tipo de entorno` → `GPU T4`): 10-30s por frase. Sin GPU: bastantes minutos.

**Pasos:** 1) Celda 1 (instalación, ~2-4 min) → 2) Celda 2 (idioma) → 3) Celda 3 (interfaz).

> 🎤 **Micrófono:** cuando ejecutes la Celda 3 verás un enlace `https://xxxxx.gradio.live`. **Ábrelo en una pestaña nueva** (no uses el recuadro embebido) — así el navegador sí permite usar el micrófono. Dentro de esa pestaña hay un botón "🔓 Habilitar micrófono".
> 🎯 **Consistencia:** usa "Modo estable" o fija una semilla para que el mismo audio + texto den siempre el mismo resultado.
> 💡 Si algo falla: vuelve a ejecutar la Celda 1, luego la 2 y la 3.

In [ ]:
#@title 🔧 Celda 1: Instalación de dependencias (ejecutar una sola vez)
#@markdown Tarda 2-4 min. Espera el mensaje `✅ Todo listo`.

import sys, subprocess, warnings
warnings.filterwarnings("ignore")

# qwen-tts (Qwen3-TTS) en vez de coqui-tts/XTTS: compatible con las
# versiones de numpy/scipy/transformers que ya trae Colab. No fijamos
# versiones a la fuerza para no dejar el entorno inconsistente.
# faster-whisper transcribe el audio de referencia automáticamente.

print("📦 Instalando dependencias de script1clic.com...")
print("⏳ Esto puede tardar unos minutos, por favor espera...\n")

paquetes = [
    "qwen-tts",
    "soundfile",
    "gradio",
    "faster-whisper",
]

resultado = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U"] + paquetes,
    capture_output=True, text=True
)

subprocess.run(["apt-get", "-y", "-qq", "install", "ffmpeg"], check=False,
               capture_output=True, text=True)

if resultado.returncode != 0:
    print("⚠️ Ocurrió un problema durante la instalación.\n")
    print(resultado.stderr[-1500:])
    print("\n👉 Solución: vuelve a ejecutar esta misma Celda 1. Si el problema persiste, ve a 'Entorno de ejecución' → 'Reiniciar sesión' y vuelve a ejecutar la Celda 1.")
else:
    # Verificación en subproceso nuevo para evitar falsos positivos si
    # numpy/torch ya estaban cargados en esta sesión.
    check = subprocess.run(
        [sys.executable, "-c", "import torch, soundfile, gradio; from qwen_tts import Qwen3TTSModel; from faster_whisper import WhisperModel; print('OK')"],
        capture_output=True, text=True
    )
    if "OK" in check.stdout:
        print("✅ Todo listo. Las dependencias quedaron instaladas correctamente.")
        print("👉 Ahora ejecuta la Celda 2.")
    else:
        print("⚠️ Las dependencias se instalaron pero algo no importa correctamente.")
        print(check.stderr[-1500:])
        print("\n👉 Solución: ve a 'Entorno de ejecución' → 'Reiniciar sesión' y vuelve a ejecutar la Celda 1.")


In [ ]:
#@title 🎛️ Celda 2: Configuración (elige tus opciones)

#@markdown ### 🔊 Idioma de la voz clonada
idioma = "es" #@param ["es", "en", "fr", "de", "it", "pt", "ru", "ja", "ko", "zh-cn"]

#@markdown ### 🌐 Compartir enlace público (útil si usas móvil o compartes con otros)
compartir_publico = True #@param {type:"boolean"}

# Qwen3-TTS soporta 10 idiomas principales; mapeamos el código corto al
# nombre completo que pide la librería.
MAPA_IDIOMAS = {
    "es": "Spanish",
    "en": "English",
    "fr": "French",
    "de": "German",
    "it": "Italian",
    "pt": "Portuguese",
    "ru": "Russian",
    "ja": "Japanese",
    "ko": "Korean",
    "zh-cn": "Chinese",
}
idioma_qwen = MAPA_IDIOMAS[idioma]

print("✅ Configuración guardada:")
print(f"   Idioma: {idioma} ({idioma_qwen})")
print(f"   Enlace público: {'Sí' if compartir_publico else 'No'}")
print("\n👉 Ahora ejecuta la Celda 3 para abrir la interfaz.")

In [ ]:
#@title 🚀 Celda 3: Iniciar Clonador de Voz (interfaz visual)

import os, traceback, tempfile
import numpy as np
import warnings
warnings.filterwarnings("ignore")

MENSAJE_ERROR_AMIGABLE = """
⚠️ **Algo salió mal.** Suele deberse a: audio de referencia muy corto (usa 5-15s, limpio), texto de referencia que no coincide con el audio, o texto a generar vacío.

👉 Verifica y vuelve a intentarlo. Si persiste, re-ejecuta la Celda 1, luego la 2 y la 3.
"""

MENSAJE_DEPENDENCIAS_FALTANTES = """
⚠️ **Faltan dependencias.** Ejecuta la Celda 1, espera `✅ Todo listo`, y luego la Celda 2 y la 3.
"""

try:
    import torch
    import soundfile as sf
    from qwen_tts import Qwen3TTSModel
    from faster_whisper import WhisperModel
    dependencias_ok = True
except Exception:
    dependencias_ok = False
    print(MENSAJE_DEPENDENCIAS_FALTANTES)
    print("\n---- Detalle técnico ----")
    traceback.print_exc()

if dependencias_ok:
    try:
        import gradio as gr

        dispositivo = "cuda:0" if torch.cuda.is_available() else "cpu"
        dtype = torch.bfloat16 if dispositivo.startswith("cuda") else torch.float32
        print(f"🖥️ Usando: {dispositivo.upper()}")

        print("📥 Cargando modelo de clonación de voz (Qwen3-TTS)...")
        modelo_tts = Qwen3TTSModel.from_pretrained(
            "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
            device_map=dispositivo,
            dtype=dtype,
        )
        print("✅ Modelo de voz cargado.\n")

        tamaño_whisper = "small" if dispositivo.startswith("cuda") else "base"
        compute_type = "float16" if dispositivo.startswith("cuda") else "int8"
        print(f"📥 Cargando Whisper '{tamaño_whisper}'...")
        modelo_whisper = WhisperModel(
            tamaño_whisper,
            device="cuda" if dispositivo.startswith("cuda") else "cpu",
            compute_type=compute_type,
        )
        print("✅ Modelo de transcripción cargado.\n")

        def recortar_silencios(ruta_audio, umbral_db=-40, padding_seg=0.05):
            """Quita silencio de arranque/final del audio de referencia."""
            try:
                data, sr = sf.read(ruta_audio)
                mono = data.mean(axis=1) if data.ndim > 1 else data
                amplitud = np.abs(mono)
                pico = np.max(amplitud) if amplitud.size else 0
                if pico <= 0:
                    return ruta_audio
                umbral = (10 ** (umbral_db / 20)) * pico
                indices = np.where(amplitud > umbral)[0]
                if indices.size == 0:
                    return ruta_audio
                pad = int(padding_seg * sr)
                inicio = max(0, indices[0] - pad)
                fin = min(len(mono), indices[-1] + pad)
                if fin - inicio < sr * 0.5:
                    return ruta_audio
                recortado = data[inicio:fin]
                nueva_ruta = tempfile.mktemp(suffix=".wav")
                sf.write(nueva_ruta, recortado, sr)
                return nueva_ruta
            except Exception:
                traceback.print_exc()
                return ruta_audio

        def transcribir_audio(audio_referencia, modo_solo_timbre_check=False):
            if audio_referencia is None:
                yield ""
                return
            if modo_solo_timbre_check:
                # En "Solo timbre" no se necesita transcripción -> nos ahorramos
                # todo el paso de Whisper al subir/grabar el audio.
                yield "(no requerido en modo 'Solo timbre')"
                return
            yield "🔄 Transcribiendo..."
            try:
                audio_limpio = recortar_silencios(audio_referencia)
                segmentos, _info = modelo_whisper.transcribe(audio_limpio, beam_size=5)
                texto_detectado = " ".join(seg.text.strip() for seg in segmentos).strip()
                yield texto_detectado if texto_detectado else "⚠️ No se pudo transcribir. Prueba con otro audio."
            except Exception:
                traceback.print_exc()
                yield "⚠️ No se pudo transcribir. Prueba con otro audio."

        def clonar_voz(audio_referencia, texto_referencia, texto, idioma_sel,
                       semilla, modo_estable, temperatura, top_p,
                       modo_solo_timbre):
            try:
                if audio_referencia is None:
                    return None, "⚠️ Sube o graba un audio de referencia primero."

                audio_limpio = recortar_silencios(audio_referencia)

                if (not texto_referencia or texto_referencia.strip() == ""
                        or texto_referencia.startswith("⚠️") or texto_referencia.startswith("🔄")):
                    try:
                        segmentos, _info = modelo_whisper.transcribe(audio_limpio, beam_size=5)
                        texto_referencia = " ".join(seg.text.strip() for seg in segmentos).strip()
                    except Exception:
                        texto_referencia = ""
                    # En modo "solo timbre" no hace falta la transcripción para clonar,
                    # así que no bloqueamos aquí si no se pudo transcribir.
                    if not texto_referencia and not modo_solo_timbre:
                        return None, "⚠️ No se pudo transcribir el audio. Prueba con uno más claro, o activa 'Solo timbre' (no necesita transcripción)."

                if not texto or texto.strip() == "":
                    return None, "⚠️ Escribe el texto que quieres que diga la voz clonada."

                aviso_duracion = ""
                try:
                    info_audio = sf.info(audio_limpio)
                    duracion = info_audio.frames / info_audio.samplerate
                    if duracion < 3:
                        aviso_duracion = " ⚠️ Audio de referencia muy corto; prueba 5-15s."
                    elif duracion > 20:
                        aviso_duracion = " ⚠️ Con 5-15s de referencia suele bastar."
                except Exception:
                    pass

                # Semilla fija -> mismo audio+texto+semilla dan siempre el mismo resultado.
                semilla_int = int(semilla) if semilla is not None else 42
                torch.manual_seed(semilla_int)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(semilla_int)

                kwargs_generacion = dict(do_sample=not modo_estable)
                if not modo_estable:
                    kwargs_generacion["temperature"] = float(temperatura)
                    kwargs_generacion["top_p"] = float(top_p)

                with torch.inference_mode():
                    voice_clone_prompt = modelo_tts.create_voice_clone_prompt(
                        ref_audio=audio_limpio,
                        ref_text=texto_referencia if not modo_solo_timbre else None,
                        x_vector_only_mode=modo_solo_timbre,
                    )

                    wavs, sr = modelo_tts.generate_voice_clone(
                        text=texto.strip(),
                        language=idioma_sel,
                        voice_clone_prompt=voice_clone_prompt,
                        **kwargs_generacion,
                    )
                audio_final = wavs[0]

                salida_path = tempfile.mktemp(suffix=".wav")
                sf.write(salida_path, audio_final, sr)
                nota_modo = " (modo estable)" if modo_estable else f" (semilla {semilla_int})"
                nota_timbre = " (solo timbre)" if modo_solo_timbre else ""
                return salida_path, "✅ ¡Listo!" + aviso_duracion + nota_modo + nota_timbre

            except Exception as e:
                print("---- ERROR TÉCNICO ----")
                traceback.print_exc()
                return None, MENSAJE_ERROR_AMIGABLE

        # JS: fuerza el permiso de micrófono con un clic real del usuario.
        JS_HABILITAR_MIC = """
        async () => {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
                stream.getTracks().forEach(t => t.stop());
                return "✅ Micrófono habilitado. Ya puedes grabar arriba.";
            } catch (err) {
                return "⚠️ Bloqueado (" + err.message + "). Abre el enlace https://xxxxx.gradio.live en una pestaña nueva (no el recuadro embebido) y reintenta.";
            }
        }
        """

        with gr.Blocks(title="Clonador de Voz - script1clic.com", theme=gr.themes.Soft()) as interfaz:
            gr.Markdown("""
            # 🎙️ Clonador de Voz con IA (Qwen3-TTS)
            ### Powered by script1clic.com

            1. Sube o **graba** (botón 🔓 primero si usas el micrófono) un audio de 5-15s, limpio.
            2. Revisa/corrige el texto detectado si Whisper se equivocó.
            3. Escribe el texto nuevo y pulsa "Generar Voz Clonada".

            ℹ️ Mic no detectado → abre el enlace `gradio.live` en pestaña nueva. Voz distinta al original → activa "Solo timbre" en Opciones avanzadas.
            """)

            with gr.Row():
                estado_mic = gr.Textbox(label="🎙️ Estado del micrófono", interactive=False, value="Aún no verificado. Pulsa el botón →")
                boton_mic = gr.Button("🔓 Habilitar micrófono", scale=0)
            boton_mic.click(fn=None, inputs=None, outputs=estado_mic, js=JS_HABILITAR_MIC)

            with gr.Row():
                with gr.Column():
                    audio_input = gr.Audio(
                        label="🎤 Audio de referencia (sube o graba)",
                        type="filepath",
                        sources=["upload", "microphone"]
                    )
                    texto_referencia_input = gr.Textbox(
                        label="📄 Texto detectado (revisa y corrige si Whisper se equivocó)",
                        placeholder="Se completa solo al subir/grabar el audio. Puedes editarlo si no es exacto.",
                        lines=2, max_lines=4, interactive=True
                    )
                    texto_input = gr.Textbox(
                        label="📝 Texto nuevo a generar",
                        placeholder="Escribe aquí el texto...",
                        lines=4
                    )

                    with gr.Accordion("⚙️ Consistencia de la voz", open=False):
                        modo_solo_timbre_input = gr.Checkbox(
                            label="🎯 Solo timbre (más fiel al audio original, ignora la transcripción; menos expresivo)",
                            value=False
                        )
                        modo_estable_input = gr.Checkbox(
                            label="Modo estable (recomendado)", value=True
                        )
                        semilla_input = gr.Number(
                            label="🔒 Semilla (repetible con el mismo audio+texto)",
                            value=42, precision=0
                        )
                        temperatura_input = gr.Slider(
                            label="Temperatura (solo sin modo estable)",
                            minimum=0.1, maximum=1.2, value=0.7, step=0.05
                        )
                        top_p_input = gr.Slider(
                            label="Top-p (solo sin modo estable)",
                            minimum=0.1, maximum=1.0, value=0.9, step=0.05
                        )
                        gr.Markdown("💡 Reutiliza el mismo audio de referencia entre generaciones. Si la voz sale distinta al original, prueba 'Solo timbre'.")

                    boton_generar = gr.Button("✨ Generar Voz Clonada", variant="primary")

                with gr.Column():
                    audio_output = gr.Audio(label="🔊 Resultado", type="filepath")
                    estado_output = gr.Textbox(label="📋 Estado", interactive=False)

            audio_input.change(fn=transcribir_audio, inputs=[audio_input, modo_solo_timbre_input], outputs=texto_referencia_input)
            modo_solo_timbre_input.change(fn=transcribir_audio, inputs=[audio_input, modo_solo_timbre_input], outputs=texto_referencia_input)

            boton_generar.click(
                fn=clonar_voz,
                inputs=[audio_input, texto_referencia_input, texto_input, gr.State(idioma_qwen),
                        semilla_input, modo_estable_input, temperatura_input, top_p_input,
                        modo_solo_timbre_input],
                outputs=[audio_output, estado_output]
            )

            gr.Markdown("---\n*script1clic.com*")

        if not compartir_publico:
            print("⚠️ Activa 'compartir_publico = True' en la Celda 2 para que el micrófono funcione bien.")

        print("\n👉 Abre el enlace https://xxxxx.gradio.live en una PESTAÑA NUEVA (no el recuadro embebido) para que el micrófono funcione.\n")

        interfaz.launch(share=True, debug=False, inline=False)

    except Exception as e:
        print(MENSAJE_ERROR_AMIGABLE)
        print("\n---- Detalle técnico ----")
        traceback.print_exc()
